# Milestone 2 RAG Exploration

This notebook explores the Milestone 2 backend workflow for the Amazon Reviews 2023 **Video_Games** project.

It includes:

- LLM/API setup and sanity checks
- semantic retrieval + semantic RAG
- hybrid retrieval + hybrid RAG
- prompt variant comparison
- saved example outputs for later discussion and UI handoff

This notebook assumes:
- raw data and processed Milestone 1 sample artifacts already exist
- a local `.env` file exists in the project root
- the `.env` file contains:
  - `GROQ_API_KEY`
  - `LLM_MODEL` (for example `llama-3.1-8b-instant`)


## Workflow diagram (matches README)

One **RAG mode** per request: **Semantic RAG** uses dense retrieval only; **Hybrid RAG** runs BM25 and semantic in parallel and merges with **RRF**. Both paths share **build_context** → prompt → **Groq**.

```mermaid
flowchart TD
    Q[User query]
    Q --> MODE{RAG mode}
    MODE -->|Semantic RAG| SEM["SemanticRetriever (FAISS + embeddings)"]
    SEM --> DOCS[Top-k review rows]
    MODE -->|Hybrid RAG| BM[BM25Retriever]
    MODE -->|Hybrid RAG| SEM2[SemanticRetriever]
    BM --> RRF[Reciprocal rank fusion]
    SEM2 --> RRF
    RRF --> DOCS
    DOCS --> CTX["build_context (ASIN, title, rating, review text)"]
    CTX --> PROMPT[System prompt + context + question]
    PROMPT --> GROQ[ChatGroq on Groq API]
    GROQ --> ANS[Generated answer]
```


In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import os
import pandas as pd
from dotenv import load_dotenv

load_dotenv(dotenv_path=PROJECT_ROOT / ".env")

from langchain_groq import ChatGroq

from src.semantic import SemanticRetriever
from src.bm25 import BM25Retriever
from src.hybrid import HybridRetriever
from src.rag_pipeline import SemanticRAGPipeline, HybridRAGPipeline

/Users/jiafuli/miniforge3/envs/dsci575-ml/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Paths and configuration

In [2]:
processed_dir = PROJECT_ROOT / "data" / "processed"

CORPUS_PATH = processed_dir / "video_games_corpus_sample.csv"
BM25_INDEX_PATH = processed_dir / "bm25_sample_index.pkl"
BM25_TOKENS_PATH = processed_dir / "bm25_sample_tokens.pkl"
FAISS_INDEX_PATH = processed_dir / "faiss_sample.index"
SEMANTIC_METADATA_PATH = processed_dir / "semantic_sample_metadata.pkl"

LLM_MODEL = os.getenv("LLM_MODEL", "llama-3.1-8b-instant")

print("Processed dir:", processed_dir)
print("LLM model:", LLM_MODEL)

Processed dir: /Users/jiafuli/Desktop/Project/575_ml/data/processed
LLM model: llama-3.1-8b-instant


## Check that Milestone 1 artifacts exist

In [3]:
required_files = [
    CORPUS_PATH,
    BM25_INDEX_PATH,
    BM25_TOKENS_PATH,
    FAISS_INDEX_PATH,
    SEMANTIC_METADATA_PATH,
]

for path in required_files:
    print(path.name, "->", path.exists())

video_games_corpus_sample.csv -> True
bm25_sample_index.pkl -> True
bm25_sample_tokens.pkl -> True
faiss_sample.index -> True
semantic_sample_metadata.pkl -> True


## API key sanity check

In [4]:
print("Has GROQ_API_KEY:", bool(os.getenv("GROQ_API_KEY")))
print("Model from env:", os.getenv("LLM_MODEL"))

Has GROQ_API_KEY: True
Model from env: llama-3.1-8b-instant


## Test the Groq API directly

In [5]:
llm = ChatGroq(
    model=LLM_MODEL,
    api_key=os.getenv("GROQ_API_KEY"),
)

response = llm.invoke("Say hello in one short sentence.")
print(response.content)

Hello.


## Model choice and rationale

For Milestone 2, we use a hosted LLM through the Groq API instead of running a local model.
This is useful because:

- it avoids local model downloads and hardware constraints
- it keeps setup lighter for a laptop-based project
- it lets us focus on retrieval, prompts, and grounded generation

The current model is controlled by the `.env` variable `LLM_MODEL`.


## Test the semantic retriever only

In [6]:
semantic_retriever = SemanticRetriever.load_saved(
    index_path=FAISS_INDEX_PATH,
    metadata_path=SEMANTIC_METADATA_PATH,
)

semantic_docs = semantic_retriever.search(
    "story-rich scary game with dark atmosphere",
    top_k=5,
)

semantic_docs[["product_title", "review_title", "rating", "semantic_score"]].head()

/Users/jiafuli/miniforge3/envs/dsci575-ml/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


,product_title,review_title,rating,semantic_score
0,Among the Sleep: Enhanced Edition - Nintendo S...,Hated this game.,1.0,0.557222
1,Scratches: Directors Cut - PC,A haunting vacation you'll never forget...,4.0,0.492031
2,Bad Mojo: Redux - PC,"A gritty, edgy homage to Kafka and Lynch that ...",5.0,0.487693
3,Broken Sword: Sleeping Dragon - PC,George and Nico are back in style!,5.0,0.485437
4,Silent Hill: Downpour - Xbox 360,Never finished it,3.0,0.469734


## Semantic RAG pipeline

In [7]:
semantic_rag = SemanticRAGPipeline(
    corpus_path=str(CORPUS_PATH),
    faiss_index_path=str(FAISS_INDEX_PATH),
    metadata_path=str(SEMANTIC_METADATA_PATH),
    model_name=LLM_MODEL,
    top_k=5,
)

/Users/jiafuli/miniforge3/envs/dsci575-ml/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


### Test semantic RAG with one query

In [8]:
semantic_result = semantic_rag.answer("story-rich scary game with dark atmosphere")

print("=== SEMANTIC RAG ANSWER ===")
print(semantic_result["answer"])

=== SEMANTIC RAG ANSWER ===
Based on the provided context, I would recommend the following games that match your criteria:

1. Scratches: Directors Cut (ASIN: B000KMCF0G) - This game is described as "dark" with a "foreboding atmosphere" and is rated 4.0. The review text mentions a sense of unease and fear, with sounds that can't be explained away and a mysterious storyline.
2. Bad Mojo: Redux (ASIN: B0006AAOJQ) - This game is described as "gritty" and "edgy" with a "dark" atmosphere, and is rated 5.0. The review text mentions the game's ability to create a sense of unease and fear, with a focus on a cockroach's perspective.
3. Broken Sword: Sleeping Dragon (ASIN: B0000A3442) is not a story-rich scary game with a dark atmosphere, although it's a great game as a whole.

However, if I must choose one game, I would recommend Scratches: Directors Cut (ASIN: B000KMCF0G) as it appears to match your criteria the most.


In [9]:
semantic_result["docs"][["product_title", "review_title", "rating"]].head()

,product_title,review_title,rating
0,Among the Sleep: Enhanced Edition - Nintendo S...,Hated this game.,1.0
1,Scratches: Directors Cut - PC,A haunting vacation you'll never forget...,4.0
2,Bad Mojo: Redux - PC,"A gritty, edgy homage to Kafka and Lynch that ...",5.0
3,Broken Sword: Sleeping Dragon - PC,George and Nico are back in style!,5.0
4,Silent Hill: Downpour - Xbox 360,Never finished it,3.0


In [10]:
print("=== CONTEXT PREVIEW ===")
print(semantic_result["context"][:1500])

=== CONTEXT PREVIEW ===
ASIN: B07NDZZHPF
Title: Among the Sleep: Enhanced Edition - Nintendo Switch
Rating: 1.0
Review title: Hated this game.
Review text: Bad graphics and questionable storyline. You are a baby escaping an abusive mother. It was just weird and not fun.


ASIN: B000KMCF0G
Title: Scratches: Directors Cut - PC
Rating: 4.0
Review title: A haunting vacation you'll never forget...
Review text: In the independently developed Scratches, you play as Michael Arthate, a British horror writer who's hard-pressed to finish his sophomore novel. In an attempt to seek inspiration, you arrange to stay at a dilapidated Victorian manner in the English countryside. The next three days will change your life forever. If I had to sum up Scratches in one word, it would be dark. Dark ambiance, dark motivations, dark secrets await you.<br /><br />The game is set in the year 1976, and during the course of your investigations you'll revisit the shocking past of Blackwood Manor. Feverish dreams (o

## Prompt variants

In [11]:
SYSTEM_PROMPT_V1 = """
You are a helpful Amazon shopping assistant.
Answer the question using ONLY the provided context from Amazon product reviews and metadata.
Do not make up product details that are not in the context.
When possible, mention the product title or ASIN.
"""

SYSTEM_PROMPT_V2 = """
You are a careful product recommendation assistant.
Use only the retrieved Amazon review context below.
If the context is insufficient, say so clearly.
Keep the answer concise, helpful, and grounded in the retrieved evidence.
"""

SYSTEM_PROMPT_V3 = """
You are an Amazon reviews analyst.
Answer the user only from the retrieved reviews and product metadata.
Prefer direct evidence from the retrieved context, and avoid unsupported claims.
"""

In [12]:
semantic_rag_v1 = SemanticRAGPipeline(
    corpus_path=str(CORPUS_PATH),
    faiss_index_path=str(FAISS_INDEX_PATH),
    metadata_path=str(SEMANTIC_METADATA_PATH),
    model_name=LLM_MODEL,
    top_k=5,
    system_prompt=SYSTEM_PROMPT_V1,
)

semantic_rag_v2 = SemanticRAGPipeline(
    corpus_path=str(CORPUS_PATH),
    faiss_index_path=str(FAISS_INDEX_PATH),
    metadata_path=str(SEMANTIC_METADATA_PATH),
    model_name=LLM_MODEL,
    top_k=5,
    system_prompt=SYSTEM_PROMPT_V2,
)

semantic_rag_v3 = SemanticRAGPipeline(
    corpus_path=str(CORPUS_PATH),
    faiss_index_path=str(FAISS_INDEX_PATH),
    metadata_path=str(SEMANTIC_METADATA_PATH),
    model_name=LLM_MODEL,
    top_k=5,
    system_prompt=SYSTEM_PROMPT_V3,
)

/Users/jiafuli/miniforge3/envs/dsci575-ml/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/Users/jiafuli/miniforge3/envs/dsci575-ml/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/Users/jiafuli/miniforge3/envs/dsci575-ml/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [13]:
test_query = "best racing game with fun tracks"

ans_v1 = semantic_rag_v1.answer(test_query)["answer"]
ans_v2 = semantic_rag_v2.answer(test_query)["answer"]
ans_v3 = semantic_rag_v3.answer(test_query)["answer"]

print("=== Prompt V1 ===")
print(ans_v1)
print("\n=== Prompt V2 ===")
print(ans_v2)
print("\n=== Prompt V3 ===")
print(ans_v3)

=== Prompt V1 ===
Based on the provided context, the Cars 3: Driven to Win - Xbox 360 (ASIN: B06Y6GPGQ8) seems to be a good option. The reviewer mentions that the game has "a good variety of characters, tracks, and types of activities" and that the tracks have "fun shortcuts to explore".

=== Prompt V2 ===
Based on the retrieved Amazon review context, I recommend the following racing games with fun tracks:

1. Cars 3: Driven to Win - Xbox 360: Reviewer praises the variety of characters, tracks, and activities, with fun shortcuts and practice maps to explore.
2. Wreckfest (PS5): Reviewer states that the game has many different races, so you won't get bored.

These games seem to offer the most variety and fun track features among the options listed in the context.

=== Prompt V3 ===
Based on the provided reviews and product metadata, I would recommend "Cars 3: Driven to Win - Xbox 360" (ASIN: B06Y6GPGQ8) and "Wreckfest (PS5)" (ASIN: B091BYX2HY).

"Cars 3: Driven to Win" has been praised 

### Prompt observation notes

**Setup:** Same query (`best racing game with fun tracks`), same top-5 retrieval, three system prompts (V1–V3) and printed answers in the cell above.

**Most grounded:** **V2 and V3** stay closest to review wording (tracks, shortcuts, variety). **V1** is also grounded but only surfaces **Cars 3** and omits **Wreckfest**, which appears in the same retrieved context and is used by V2/V3—so V1 is narrower, not wrong, but less complete relative to the evidence set.

**Most concise:** **V1** (short paragraph, single product). **V2** is the best balance of brevity and coverage (short numbered list with two products). **V3** is the longest because it adds explicit discussion of a **non-match** (Phar Lap).

**Unsupported claims:** **V3** performs best: it explicitly **rejects** Phar Lap as not fitting the “fun tracks” brief, which matches the need to avoid over-recommending from weak lexical overlap. **V2** avoids inventing details and lists only what reviews support. **V1** does not invent facts but under-uses available evidence (single pick).

**Takeaway:** We keep **V1-style** instructions in production code for simplicity; for reports, **V2** is preferable for concise multi-item answers and **V3** when the query invites excluding borderline hits.


## Test the hybrid retriever only

In [14]:
bm25_retriever = BM25Retriever.load_saved(
    corpus_path=CORPUS_PATH,
    index_path=BM25_INDEX_PATH,
    tokens_path=BM25_TOKENS_PATH,
)

semantic_retriever = SemanticRetriever.load_saved(
    index_path=FAISS_INDEX_PATH,
    metadata_path=SEMANTIC_METADATA_PATH,
)

hybrid_retriever = HybridRetriever(
    bm25_retriever=bm25_retriever,
    semantic_retriever=semantic_retriever,
    top_k=5,
)

hybrid_docs = hybrid_retriever.search(
    "story-rich scary game with dark atmosphere",
    top_k=5,
)

hybrid_docs[["product_title", "review_title", "rating", "hybrid_score"]].head()

/Users/jiafuli/miniforge3/envs/dsci575-ml/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


,product_title,review_title,rating,hybrid_score
0,Fatal Frame 2: Crimson Butterfly,Good luck finishing this yourself,5.0,0.016393
1,Among the Sleep: Enhanced Edition - Nintendo S...,Hated this game.,1.0,0.016393
2,Little Nightmares - Xbox One [Digital Code],Reminds me of Dark Eye from the PC.,4.0,0.016129
3,Scratches: Directors Cut - PC,A haunting vacation you'll never forget...,4.0,0.016129
4,Among the Sleep - PlayStation 4,Inserting Game!,5.0,0.015873


## Hybrid RAG pipeline

In [15]:
hybrid_rag = HybridRAGPipeline(
    corpus_path=str(CORPUS_PATH),
    bm25_index_path=str(BM25_INDEX_PATH),
    bm25_tokens_path=str(BM25_TOKENS_PATH),
    faiss_index_path=str(FAISS_INDEX_PATH),
    metadata_path=str(SEMANTIC_METADATA_PATH),
    model_name=LLM_MODEL,
    top_k=5,
)

/Users/jiafuli/miniforge3/envs/dsci575-ml/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [16]:
hybrid_result = hybrid_rag.answer("story-rich scary game with dark atmosphere")

print("=== HYBRID RAG ANSWER ===")
print(hybrid_result["answer"])

=== HYBRID RAG ANSWER ===
Based on the Amazon reviews, I would recommend the following games that match your criteria:

1. **Scratches: Directors Cut - PC** (B000KMCF0G) - This game is described as a "haunting vacation" with a dark atmosphere, and the reviewer praises its ability to create a foreboding and evil atmosphere without gore. The game's story is also well-received, and the reviewer mentions that the pacing is brilliant.
2. **Fatal Frame 2: Crimson Butterfly** (B0000AI1KK) - This game is described as one of the best survival horror games out there, with a great atmosphere and a story that ties well with the setting. The reviewer praises the developers' ability to create a tense and scary experience.
3. **Scratches** is also similar to **Barrow Hill: Curse of the Ancient Circle** which is recommended after Scratches, hence this game might also be a good fit, but we do not have the ASIN for this title in the provided context.

Note that **Among the Sleep** and **Little Nightmare

In [17]:
hybrid_result["docs"][["product_title", "review_title", "rating"]].head()

,product_title,review_title,rating
0,Fatal Frame 2: Crimson Butterfly,Good luck finishing this yourself,5.0
1,Among the Sleep: Enhanced Edition - Nintendo S...,Hated this game.,1.0
2,Little Nightmares - Xbox One [Digital Code],Reminds me of Dark Eye from the PC.,4.0
3,Scratches: Directors Cut - PC,A haunting vacation you'll never forget...,4.0
4,Among the Sleep - PlayStation 4,Inserting Game!,5.0


## Compare semantic RAG vs hybrid RAG

In [18]:
comparison_query = "story-rich scary game with dark atmosphere"

if "semantic_result" not in globals():
    semantic_result = semantic_rag.answer(comparison_query)

if "hybrid_result" not in globals():
    hybrid_result = hybrid_rag.answer(comparison_query)

print("=== SEMANTIC RAG ===")
print(semantic_result["answer"])

print("\n=== HYBRID RAG ===")
print(hybrid_result["answer"])

=== SEMANTIC RAG ===
Based on the provided context, I would recommend the following games that match your criteria:

1. Scratches: Directors Cut (ASIN: B000KMCF0G) - This game is described as "dark" with a "foreboding atmosphere" and is rated 4.0. The review text mentions a sense of unease and fear, with sounds that can't be explained away and a mysterious storyline.
2. Bad Mojo: Redux (ASIN: B0006AAOJQ) - This game is described as "gritty" and "edgy" with a "dark" atmosphere, and is rated 5.0. The review text mentions the game's ability to create a sense of unease and fear, with a focus on a cockroach's perspective.
3. Broken Sword: Sleeping Dragon (ASIN: B0000A3442) is not a story-rich scary game with a dark atmosphere, although it's a great game as a whole.

However, if I must choose one game, I would recommend Scratches: Directors Cut (ASIN: B000KMCF0G) as it appears to match your criteria the most.

=== HYBRID RAG ===
Based on the Amazon reviews, I would recommend the following ga

### Comparison notes

**Setup:** Same query (`story-rich scary game with dark atmosphere`), semantic RAG vs hybrid RAG; outputs are printed above and saved as `milestone2_semantic_docs_preview.csv`, `milestone2_hybrid_docs_preview.csv`, and matching `.txt` answers in the “Save example outputs” cell below.

**Retrieved documents:** **Hybrid** top-5 skews toward **survival / horror** titles (e.g. Fatal Frame 2, Scratches, Little Nightmares) via BM25 lexical signal plus dense similarity. **Semantic** top-5 still includes horror-adjacent items but can rank **story-rich** phrasing in ways that pull in **less on-theme** games (e.g. Broken Sword appears with a caveat in the semantic answer). For this query, **hybrid produced more thematically aligned supporting rows** for “scary / dark atmosphere.”

**Answer completeness:** The **hybrid** answer is **more complete**: it discusses multiple strong matches (Scratches, Fatal Frame 2) with fuller evidence from reviews and hedges about titles without ASINs. The **semantic** answer lists three games but one entry is partly a **negative** suitability note (Broken Sword), which reduces useful breadth for the stated criteria.

**Grounding:** **Hybrid** stays tighter to horror/survival framing from the retrieved blurbs. **Semantic** remains factual but includes a weaker fit (Broken Sword called out as not really matching “scary”), which is **honest** yet shows retrieval noise on a small corpus. **Caveat from this run:** the hybrid **LLM** answer mentions **Barrow Hill** even though that title is **not** in the printed top-5 rows — an example of **unsupported name-dropping** despite better retrieval.

**Takeaway:** Hybrid helped retrieval for genre keywords + paraphrase on this query; we still need strict “only names from context” behavior in production prompts. We report both pipelines in Milestone 2 and use the saved CSV/txt previews as evidence.


## Save example outputs

In [19]:
semantic_result["docs"].to_csv(processed_dir / "milestone2_semantic_docs_preview.csv", index=False)
hybrid_result["docs"].to_csv(processed_dir / "milestone2_hybrid_docs_preview.csv", index=False)

with open(processed_dir / "milestone2_semantic_answer.txt", "w", encoding="utf-8") as f:
    f.write(semantic_result["answer"])

with open(processed_dir / "milestone2_hybrid_answer.txt", "w", encoding="utf-8") as f:
    f.write(hybrid_result["answer"])

print("Saved example outputs to data/processed/")

Saved example outputs to data/processed/


## Milestone 2 exploration summary (results from this notebook)

- **API:** Groq + `llama-3.1-8b-instant` responds reliably for short sanity prompts; RAG quality is dominated by **retrieval fit** and corpus size, not API flakiness.
- **Semantic vs hybrid retrieval:** On our fixed horror query, **hybrid** top-5 ranked more genre-appropriate rows (Fatal Frame, Scratches, Little Nightmares) than **semantic** alone, which still surfaced a weaker match (Broken Sword) that the model had to caveat.
- **Prompts V1–V3:** With **identical** retrieval, **V2** gave the best concise multi-product answer; **V3** explicitly excluded borderline hits (e.g. Phar Lap); **V1** was shortest but sometimes under-used evidence — see **Prompt observation notes** above.
- **Hybrid RAG answer:** More detailed than semantic-only for the same query, but the model still **named Barrow Hill** without an ASIN in the top-5 context — we flag that as **ungrounded elaboration** in **Comparison notes**.
- **Saved artifacts:** `milestone2_semantic_docs_preview.csv`, `milestone2_hybrid_docs_preview.csv`, `milestone2_semantic_answer.txt`, `milestone2_hybrid_answer.txt` under `data/processed/` for reports and UI alignment.
